# ⚠️ Required First Step — Enable GPU

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU** (free) or **A100** if available
4. Click **Save**

Then: **Runtime → Run all** (Ctrl+F9 / Cmd+F9)

---

# TunedAI Labs — Causal Reasoning Demo

This notebook compares two versions of the same AI model on ten causal reasoning questions:

| Model | What it is |
|---|---|
| Base Qwen 2.5-7B | Unmodified open-source model |
| **TunedAI Labs Causal Model** | Same model, fine-tuned on causal reasoning |

Every question comes from a book or paper written **before AI existed**. The correct answer — YES or NO — was established by human experts decades ago.

**Sources:** John Snow (1854), Ignaz Semmelweis (1847), E.H. Simpson (1951), Bradford Hill (1965), David Hume (1748), R.A. Fisher (1935), Abraham Wald (1943), James Lind (1747), Francis Galton (1886), Louis Pasteur (1859)

Results save automatically — if the session restarts, run all cells again and completed questions are skipped.

---

---
## Step 1 — Install packages

Installs required libraries. Each package prints ✓ when done.

In [ ]:
import subprocess, sys

for pkg in ['transformers', 'peft', 'accelerate', 'bitsandbytes', 'openai', 'anthropic']:
    print(f"  {pkg}...", end=" ", flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print("✓" if r.returncode == 0 else f"✗  {r.stderr[:120]}")

print("\nDone.")

---
## Step 2 — API keys (optional)

Leave blank to run Base Qwen vs TunedAI only. Add keys to include GPT-4o and Claude.

In [ ]:
OPENAI_API_KEY    = ""   # optional
ANTHROPIC_API_KEY = ""   # optional

---
## Step 3 — Load models

Downloads and loads two versions of Qwen 2.5-7B. Takes about 90 seconds on A100.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import openai, anthropic

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → set T4 GPU.")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.0f} GB")

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("Loading tokenizer...", end=" ", flush=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print("✓")

if vram_gb >= 20:
    print("Loading base model float16 (~90 sec)...", end=" ", flush=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="cuda:0",
        trust_remote_code=True
    )
else:
    print("Loading base model 8-bit (~90 sec)...", end=" ", flush=True)
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
print("✓")

print("Loading tunedailabs adapter...", end=" ", flush=True)
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()
print("✓")

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("\n✓ All models ready.")

---
## Step 4 — Helper functions

Sets up the comparison engine and scoring. Runs automatically.

In [ ]:
import re, json, os, torch

SYSTEM = ("You are a careful causal reasoning expert. "
          "When asked a yes/no question, state YES or NO clearly, then explain your reasoning.")
RESULTS_FILE = "/content/tunedai_results.json"

_keywords = [
    # Q1 Simpson — NO
    [["no"], ["subgroup","stratif","confound","breakdown","each group"], ["simpson","combined","overall"]],
    # Q2 Survivorship — NO
    [["no"], ["survivorship","didn't return","not return","never returned","shot down"], ["engines","cockpit","fatal","where no holes","missing"]],
    # Q3 Hawthorne — NO
    [["no"], ["observed","being watched","attention","hawthorne","aware"], ["not lighting","behavior","monitoring"]],
    # Q4 Common cause — NO
    [["no"], ["common cause","temperature","summer","weather","heat","hot"], ["swimming","confound","third variable"]],
    # Q5 Reverse causation — NO
    [["no"], ["reverse","sicker","severity","selection"], ["not nurses","patient mix","assigned","critically ill"]],
    # Q6 Semmelweis — YES
    [["yes"], ["handwashing","prevent","caused","dropped","fell"], ["intervention","autopsy","before and after","midwives"]],
    # Q7 Snow pump — YES
    [["yes"], ["pump","contaminated water","water source"], ["intervention","handle","removed","cases fell"]],
    # Q8 Bradford Hill — YES
    [["yes"], ["smoking causes","causation","cause lung","lung cancer"], ["dose-response","quitting","temporality","9","10 times","15"]],
    # Q9 Snow natural exp — YES
    [["yes"], ["water source","contaminated","supplier"], ["natural experiment","no choice","random","eightfold","315","37"]],
    # Q10 Lind — YES
    [["yes"], ["citrus","caused","recovery","scurvy"], ["simultaneous","controlled","isolated","six days","6 days","oranges","lemons"]],
]

_results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        _results = json.load(f)
    print(f"Resuming — {len(_results)} questions already done.")

_q_counter = [0]

def _score(answer, q_idx):
    if q_idx >= len(_keywords): return 0, 3
    ans = answer.lower()
    hits = sum(1 for grp in _keywords[q_idx] if any(kw in ans for kw in grp))
    return hits, 3

def ask_local(question, use_adapter=True):
    if use_adapter: tuned_model.enable_adapter_layers()
    else:           tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=300, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(q):
    if not oai_client: return "[No OpenAI key]"
    r = oai_client.chat.completions.create(model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":q}], max_tokens=300)
    return r.choices[0].message.content.strip()

def ask_claude(q):
    if not ant_client: return "[No Anthropic key]"
    r = ant_client.messages.create(model="claude-3-5-sonnet-20241022", max_tokens=300,
        system=SYSTEM, messages=[{"role":"user","content":q}])
    return r.content[0].text.strip()

def compare_all(question, source="", answer=""):
    q_idx = _q_counter[0]; _q_counter[0] += 1
    key = str(q_idx)
    if key in _results:
        r = _results[key]
        b, t = r["base_score"], r["tuned_score"]
        print(f"[Q{q_idx+1} done — Base {b[0]}/{b[1]} · TunedAI {t[0]}/{t[1]}]")
        return
    SEP="="*70; DIV="-"*70
    print(SEP)
    if source: print(f"Source: {source}")
    if answer: print(f"Correct answer: {answer}")
    print(SEP)
    print(f"Question:\n{question}\n")
    base_ans   = ask_local(question, use_adapter=False)
    gpt_ans    = ask_gpt4(question)
    claude_ans = ask_claude(question)
    tuned_ans  = ask_local(question, use_adapter=True)
    for label, ans in [("BASE QWEN 2.5-7B",base_ans),("GPT-4o",gpt_ans),
                       ("CLAUDE 3.5 SONNET",claude_ans),("TUNEDAI ★",tuned_ans)]:
        print(DIV); print(f"[ {label} ]"); print(DIV); print(ans); print()
    b_h,b_t = _score(base_ans,q_idx); t_h,t_t = _score(tuned_ans,q_idx)
    star = " ★" if t_h>b_h else (" ↓" if t_h<b_h else "")
    print(f"  Score → Base {b_h}/{b_t}  ·  TunedAI {t_h}/{t_t}{star}\n")
    _results[key] = {"base_ans":base_ans,"tuned_ans":tuned_ans,
                     "base_score":[b_h,b_t],"tuned_score":[t_h,t_t]}
    with open(RESULTS_FILE,"w") as f: json.dump(_results,f,indent=2)

print("✓ Ready.")

---
## Question 1 — Simpson's Paradox
**Source:** E.H. Simpson, *The Interpretation of Interaction in Contingency Tables*, 1951

Treatment A outperforms Treatment B in every subgroup — but appears worse in the combined total. A classic trap in aggregate statistics.

In [ ]:
compare_all(
    question=(
        "Treatment A cures 93% of small kidney stone cases and 73% of large cases.\n"
        "Treatment B cures 87% of small cases and 69% of large cases.\n"
        "The combined success rate is 78% for A and 83% for B, because doctors assigned "
        "easier small-stone patients to Treatment B more often.\n\n"
        "Does the combined data show that Treatment B causes better patient outcomes?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source="E.H. Simpson, 1951",
    answer="NO — A is better in every subgroup. Stone size confounds the combined total. (Simpson Paradox)"
)

---
## Question 2 — Survivorship Bias
**Source:** Abraham Wald, Statistical Research Group, Columbia University, 1943

Engineers mapped bullet holes on returning Allied bombers. The military planned to armor where the holes were concentrated. Wald argued the opposite.

In [ ]:
compare_all(
    question=(
        "Engineers analyzed bullet damage on Allied bombers returning from WWII missions.\n"
        "Wings and fuselage had many bullet holes. Engines had almost none.\n"
        "They proposed adding armor to wings and fuselage — where damage was concentrated.\n\n"
        "Does the pattern of bullet holes on returning planes show that wings and fuselage need the most additional armor?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source='Abraham Wald, 1943',
    answer='NO — Planes hit in engines never returned to be counted. Armor the areas with NO holes — that is where damage is fatal. (Survivorship Bias)'
)

---
## Question 3 — The Hawthorne Effect
**Source:** Elton Mayo, Hawthorne Works studies, 1924–1932

Researchers tested whether lighting affected factory worker productivity. Every change — up or down — improved output. The variable being changed did not matter.

In [ ]:
compare_all(
    question=(
        "Researchers increased factory lighting and worker productivity rose.\n"
        "They decreased lighting back to the original level and productivity rose again.\n"
        "They decreased lighting below original and productivity still rose.\n"
        "Every change — up or down — produced higher output.\n\n"
        "Does lighting level cause the productivity improvements in these studies?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source='Elton Mayo, 1924–1932',
    answer='NO — Workers improved because they knew they were being observed. The cause was attention, not lighting. (The Hawthorne Effect)'
)

---
## Question 4 — Common Cause
**Source:** Classic epidemiological confounding example, documented in textbooks from the 1950s onward

Monthly statistics show a nearly perfect correlation between ice cream sales and drowning deaths across a decade.

In [ ]:
compare_all(
    question=(
        "Monthly data from American coastal cities, 1950–1960:\n"
        "Ice cream sales and drowning deaths both peaked every July and fell every January.\n"
        "The correlation between the two across all months was r = 0.97.\n\n"
        "Does eating ice cream cause drowning deaths?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source='Classic confounding example, 1950s',
    answer='NO — Hot summer weather causes both. Heat drives ice cream consumption and swimming activity. Ice cream and drowning share a common cause.'
)

---
## Question 5 — Reverse Causation
**Source:** Classic medical confounding example

Across hundreds of hospitals, units with more staff per patient show higher death rates. The naive reading reverses cause and effect.

In [ ]:
compare_all(
    question=(
        "Hospitals with more nurses per patient have higher patient death rates "
        "than hospitals with fewer nurses per patient.\n"
        "The pattern holds across 500 hospitals nationwide.\n\n"
        "Does a higher nurse-to-patient ratio cause more patient deaths?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source='Classic reverse causation example',
    answer='NO — Sicker patients are assigned more nurses. Severity of illness drives both staffing and deaths. The causal arrow runs the other way. (Reverse Causation)'
)

---
## Question 6 — Controlled Intervention
**Source:** Ignaz Semmelweis, Vienna General Hospital, 1847

Semmelweis mandated handwashing. Mortality in the doctors' ward collapsed immediately — and only in that ward. The midwives' ward, used as a natural control, did not change.

In [ ]:
compare_all(
    question=(
        "In 1847, doctors at Vienna General Hospital went from performing autopsies "
        "directly to delivering babies without washing hands. Maternal death rate: 10%.\n"
        "After Semmelweis mandated handwashing with chlorinated lime, the rate fell to 1.5%.\n"
        "The midwives' ward, which did not change its practice, remained at 2% throughout.\n\n"
        "Did handwashing cause the reduction in deaths in the doctors' ward?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source='Ignaz Semmelweis, 1847',
    answer='YES — Only the ward where the intervention occurred changed. The midwives ward is the control. Clear before-and-after causal evidence.'
)

---
## Question 7 — Causal Intervention
**Source:** John Snow, *On the Mode of Communication of Cholera*, 2nd edition, 1855

Snow mapped deaths, traced them to the Broad Street pump, and removed its handle. The local outbreak subsided while surrounding areas continued unaffected.

In [ ]:
compare_all(
    question=(
        "In 1854, John Snow mapped 578 cholera deaths in London and found nearly all "
        "clustered within 250 meters of the Broad Street water pump.\n"
        "He had the pump handle removed, rendering it unusable.\n"
        "New cases in the district fell sharply within days. "
        "Neighboring districts without pump removal continued to have cases.\n\n"
        "Did the Broad Street pump cause the local cholera outbreak?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source='John Snow, 1854',
    answer='YES — The pump handle removal is a controlled intervention. Cases fell immediately in the affected area only. This is causal evidence, not correlation.'
)

---
## Question 8 — Causation Without an Experiment
**Source:** Bradford Hill, *The Environment and Disease: Association or Causation?*, 1965

No randomized trial of smoking was ever run — you cannot force people to smoke for decades. Critics argued only correlation could be claimed. Bradford Hill showed the totality of evidence crossed the threshold for causation.

In [ ]:
compare_all(
    question=(
        "By 1965, 30 independent studies across multiple countries showed smokers develop "
        "lung cancer at 9 to 10 times the rate of non-smokers.\n"
        "Heavier smokers had higher rates than lighter smokers.\n"
        "Quitting smoking reduced the risk. Lung cancer developed 15 to 20 years after "
        "starting smoking — never before.\n\n"
        "Does smoking cause lung cancer?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source='Bradford Hill, 1965',
    answer='YES — Dose-response, correct temporal order, consistency across countries, and quitters showing lower rates all establish causation without an RCT.'
)

---
## Question 9 — Natural Experiment
**Source:** John Snow, *On the Mode of Communication of Cholera*, 2nd edition, 1855

Two water companies served houses on the same streets, side by side. Residents had no choice in supplier — creating a near-random assignment of water source across households.

In [ ]:
compare_all(
    question=(
        "In 1854 London, two water companies served adjacent houses on the same streets:\n"
        "Southwark & Vauxhall (drew water below sewage outfalls): 315 deaths per 10,000 houses\n"
        "Lambeth (drew water above the city, upstream of outfalls): 37 deaths per 10,000 houses\n\n"
        "Households had no choice which company supplied them — "
        "adjacent houses on the same street were served by different companies.\n\n"
        "Does water source cause the difference in cholera death rates?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source='John Snow, 1855',
    answer='YES — Random-like assignment of supplier eliminates self-selection. The eightfold difference in deaths is attributable to water contamination.'
)

---
## Question 10 — The First Controlled Trial
**Source:** James Lind, *A Treatise of the Scurvy*, 1753

Lind gave 12 sailors with scurvy 6 different treatments simultaneously. Only the citrus pair recovered. This was the first controlled clinical trial in history.

In [ ]:
compare_all(
    question=(
        "1747, HMS Salisbury. 12 sailors with advanced scurvy. Same base diet throughout.\n"
        "Six treatment pairs tested simultaneously at the same time under identical conditions:\n"
        "Cider — no recovery. Sulfuric acid — no recovery. Vinegar — no recovery.\n"
        "Seawater — no recovery. Oranges and lemons — full recovery within 6 days. "
        "Garlic paste — no recovery.\n\n"
        "Did citrus fruit cause the recovery from scurvy in the oranges-and-lemons pair?\n"
        "Answer YES or NO, then explain your reasoning."
    ),
    source='James Lind, 1753',
    answer='YES — Simultaneous comparison under identical conditions isolates citrus as the cause. All other treatments failed.'
)

---
## CLadder Verification — 5 Synthetic Questions

These questions use fictional scenarios with no pre-existing answers on the internet.
They match the exact format the model was trained on.
Scored on YES/NO correctness only.

In [ ]:
# CLadder-format synthetic questions — binary YES/NO scoring only
_cladder_qs = [
    # Rung 1 — Association
    ("In Millbrook, 12% of people who drink filtered water get stomach illness each year. "
     "68% of people who drink unfiltered water get stomach illness each year. "
     "Is drinking unfiltered water positively associated with stomach illness?",
     "yes"),
    # Rung 2 — Intervention / confounding
    ("Workers who wear safety harnesses have a 2% injury rate. "
     "Workers who do not wear harnesses have a 35% injury rate. "
     "However, safety harnesses are only issued for high-risk jobs. "
     "If we gave harnesses to all workers regardless of job risk, "
     "would the injury rate for harness wearers necessarily stay at 2%?",
     "no"),
    # Rung 2 — Intervention / do-calculus
    ("Drug X is known to cause a reduction in blood pressure. "
     "High blood pressure is known to cause stroke. "
     "If we administer Drug X to a patient, will their stroke risk decrease?",
     "yes"),
    # Rung 3 — Counterfactual
    ("A patient took antibiotic Z and recovered from a bacterial infection. "
     "Antibiotic Z is known to cause recovery from this infection. "
     "If the patient had not taken antibiotic Z, would they have recovered?",
     "no"),
    # Rung 1 — Simpson's paradox synthetic
    ("Drug A cures 90% of mild cases and 50% of severe cases. "
     "Drug B cures 80% of mild cases and 40% of severe cases. "
     "Because Drug A is given to more severe cases, its overall rate is 60% vs Drug B's 75%. "
     "Does Drug B causally produce better outcomes than Drug A?",
     "no"),
]

def _detect_yn(text):
    t = text.strip().lower()
    words = t.split()
    first = words[0].rstrip(".,!:") if words else ""
    if first == "yes": return "yes"
    if first == "no":  return "no"
    if t.startswith("yes"): return "yes"
    if t.startswith("no"):  return "no"
    return "?"

SEP = "="*70; DIV = "-"*70
print(SEP)
print("CLADDER VERIFICATION — Binary YES/NO scoring")
print("5 synthetic questions | format matches training distribution")
print(SEP)

base_correct = 0; tuned_correct = 0
for i, (q, expected) in enumerate(_cladder_qs):
    print(f"\nQ{i+1} (expected: {expected.upper()})")
    print(f"  {q[:120]}...")
    base_ans  = ask_local(q, use_adapter=False)
    tuned_ans = ask_local(q, use_adapter=True)
    b_yn = _detect_yn(base_ans);  b_ok = b_yn == expected
    t_yn = _detect_yn(tuned_ans); t_ok = t_yn == expected
    base_correct  += b_ok
    tuned_correct += t_ok
    star = " ★" if t_ok and not b_ok else (" ↓" if b_ok and not t_ok else "")
    print(f"  Base   → {b_yn.upper():3s} ({'✓' if b_ok else '✗'})  |  TunedAI → {t_yn.upper():3s} ({'✓' if t_ok else '✗'}){star}")
    print(f"  Base full: {base_ans[:120]}")
    print(f"  Tuned full: {tuned_ans[:120]}")

print(f"\n{SEP}")
print(f"  Base Qwen 2.5-7B  : {base_correct}/5 = {base_correct*20}%")
print(f"  TunedAI Labs ★    : {tuned_correct}/5 = {tuned_correct*20}%")
print(SEP)


---
## Results — All 10 Questions

Reads from saved file. Survives Colab restarts.

In [ ]:
import json, os
RESULTS_FILE = "/content/tunedai_results.json"
if not os.path.exists(RESULTS_FILE):
    print("No results yet — run the questions above first.")
else:
    with open(RESULTS_FILE) as f: res = json.load(f)
    n = len(res)
    b_total = sum(res[k]["base_score"][0] for k in res)
    t_total = sum(res[k]["tuned_score"][0] for k in res)
    maximum = n * 3
    print("=" * 70)
    print("FINAL RESULTS — TunedAI Labs Causal Reasoning Demo")
    print("10 yes/no questions · pre-AI historical sources · 3 pts each")
    print("=" * 70)
    print(f"  Base Qwen 2.5-7B (untuned) : {b_total:2d}/{maximum} = {100*b_total/maximum:.1f}%")
    print(f"  TunedAI Causal Model ★     : {t_total:2d}/{maximum} = {100*t_total/maximum:.1f}%")
    diff = 100*t_total/maximum - 100*b_total/maximum
    print(f"  Improvement                : {diff:+.1f} percentage points")
    print("=" * 70)
    print(f"\nCompleted: {n}/10\n")
    print(f"{'Q':<5} {'Base':>5} {'Tuned':>7}")
    print(f"{'----':<5} {'-----':>5} {'-------':>7}")
    for i in range(n):
        k = str(i)
        if k not in res: continue
        b_h, b_t = res[k]["base_score"]
        t_h, t_t = res[k]["tuned_score"]
        flag = " ★" if t_h > b_h else (" ↓" if t_h < b_h else "")
        print(f"Q{i+1:<4} {b_h}/{b_t}   {t_h}/{t_t}{flag}")


---
## Try Your Own Question

**Why question structure matters**

The CLadder benchmark — the test this model scored 96.96% on — is built around a specific type of question. Each question has two parts:

1. **A short factual setup** — a scenario with two groups, two outcomes, or a before-and-after observation. No interpretation, just facts.
2. **A single causal question** — one yes/no question that asks whether X *causes* Y.

This structure forces a clean separation between the evidence and the causal claim. That is exactly what causal reasoning requires: read the facts, then decide whether the relationship is causal or not.

When you write a question that follows this structure, you are testing the model on the same type of reasoning it was trained on. Open-ended or narrative questions ask it to explain or summarize — a different skill. Structured causal questions ask it to *reason*, which is where the fine-tuning shows.

---

**Template:**

> In [**setting**], [**Group A**] [**did X**].
> [**Group B**] [**did not do X** / **did Y instead**].
> [**Group A result**]. [**Group B result**].
> Does [**X**] cause [**outcome**]?

**Tips:**
- Keep the setup to 2–4 sentences
- Include only the facts needed to answer the question
- End with exactly one yes/no causal question: *Does X cause Y?*

---

**Example — fill in the blanks like this:**

> *Patients in Group A took aspirin daily. Patients in Group B took a placebo.*
> *After one year, Group A had 30% fewer heart attacks. Group B showed no change.*
> *Does daily aspirin use cause a reduction in heart attacks?*

Replace the example below with your own and run the cell.

In [ ]:
MY_QUESTION = """
Factory workers were divided into two groups. Group A received ergonomic chairs. Group B kept standard chairs.
After 6 months, Group A reported 40% less back pain. Group B reported no change.
Does using an ergonomic chair cause a reduction in back pain?
"""

compare_all(MY_QUESTION.strip(), source="Custom — your question")

---
## Share Your Results

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste what you saw.

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning tasks.
[tunedailabs.com](https://tunedailabs.com)